## Definitions and Set up

### Import packages

In [ ]:
%matplotlib widget
import typing
from time import sleep, time
import numpy as np
import scipy.signal
import asyncio
import matplotlib.pyplot as plt
from nea_tools import connect, disconnect
from scipy.optimize import curve_fit
from statistics import median
import nest_asyncio
import ipywidgets as widgets
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

### Connect to neaSCOPE

In [ ]:
loop.run_until_complete(connect())

from neaspec import context
from nea_tools.logic import scan, current_tip_position, approach_sample
from nea_tools import set_output
from nea_tools.microscope import Preview
from nea_tools.datatypes import PreviewData
import Nea.Client.SharedDefinitions as nea

### Definitions

In [ ]:
def get_cross_image(base_img, target_img):
   """Do cross-correlation on two images"""
   # get rid of the color channels by performing a grayscale transform
   # the type cast into 'float' is to avoid overflows
   if base_img.ndim == 3:
      im1_gray = np.sum(base_img.astype('float'), axis=2)
      im2_gray = np.sum(target_img.astype('float'), axis=2)
   else:
      im1_gray = base_img.astype("float")
      im2_gray = target_img.astype("float")

   # get rid of the averages, otherwise the results are not good
   im1_gray -= np.mean(im1_gray)
   im2_gray -= np.mean(im2_gray)

   # calculate the correlation image; note the flipping of onw of the images
   return scipy.signal.fftconvolve(im1_gray, im2_gray[::-1,::-1], mode='same')

def get_shift(base_img, target_img, pitch:typing.Tuple[float,float]=(1,1)) -> typing.Tuple[float,float]:
    """Returns shift of target_img to base_img. Use pitch for scaling in x and y axis. In um"""
    cross_img = get_cross_image(base_img,target_img)
    indeces = np.unravel_index(np.argmax(cross_img), cross_img.shape)
    return pitch[1]*(-indeces[1]+base_img.shape[1]/2),pitch[0]*(-indeces[0]+base_img.shape[0]/2) 

def lin(x, *param):
   """Simple linear correction of x."""
   #y = m*x+b
   return param[0]*x + param[1]

def plane_subst(z):
   """Fits a plane to data substracts data by this plane."""
   [m,n] = z.shape
   x = np.arange(0, m)
   y = np.arange(0, n)
   mx = 0
   my = 0
   for i in y: #
      #extracting one line into x direction
      zx = np.array([z[index][i] for index in x])
      #calculate slope into x direction
      #average over all slopes
      mx += curve_fit(lin, x,zx, p0=[1,0])[0][0]
   for i in x: #
      #extracting one line into y direction
      zy = z[i][:]
      #calculate slope into y direction
      #average over all slopes
      my += curve_fit(lin, y,zy, p0=[1,0])[0][0]
   mx /= len(y)
   my /= len(x)
   #Build approx. plane
   plane = np.zeros(z.shape)
   for i in range(m):
      for j in range(n):
            plane[i][j] =   mx*i + my*j
   return z-plane

def line_diff(z):
   """Applies median line difference correction to z."""
   Z = np.zeros(z.shape) #corrected image
   #corrects median difference of lines
   m = []
   m1 = []
   temp = median(z[-1][:])
   for i in range(z.shape[0]):
      Z[i][:] = z[i][:] + (temp-median(z[i][:]))
   return Z
    
class Plot:
    """
    Generates matplotlib figure intended to plot live scan data.
    Allows to add 2 independant markers to the plots.
    One marker can be set manually by clicking on the plots.
    """
    def __init__(self, img = None, num_plots = 1):
        self.fig, self.axs = plt.subplots(ncols=num_plots)
        if num_plots == 1:
            self.ax = self.axs
        self.ax_tick_labels = [47.5,52.5,47.5,52.5]
        if img is not None:
            self.update(img)
        self.markerprops = dict(
            color = "red",
            marker = "X",
            s = 50
        )
        self.titles = [""]*num_plots
        self.markerprops_2 = dict(
            color = "blue",
            marker = ".",
            s = 50
        )
        self.marker_pos = None
        self.marker_pos_2 = None
        self.fig.canvas.mpl_connect("button_press_event", self.on_mouse_clicked)
        self._on_marker_pos_changed = None
        self._do_plot = True

    def subscribe_marker_pos_changed(self, func):
        if callable(func):
            self._on_marker_pos_changed = func
    
    def update(self, img, marker_pos = None, marker_pos_2 = None, ax_id = 0, title = None, redraw = True):
        """
        Add a new image to the plt figure. 
        
        Args:
            img (np.ndarray): image to plot
            marker_pos (Tuple[int,int]): optional position of marker one
            marker_pos_2 (Tuple[int,int]): optional position of marker two
            ax_id (int): ax to plot the image to
            title (str): optional title of image displayed underneath
            redraw (bool): whether to update the whole figure.
        """
        
        if not self._do_plot:
            return
        self.axs[ax_id].cla()
        if title is not None:
            self.titles[ax_id] = str(title)
        self.axs[ax_id].set_xlabel(self.titles[ax_id])
        if "O" in self.titles[ax_id] and "P" in self.titles[ax_id]:
            cmap = "bwr"
        elif "O" in self.titles[ax_id] and "A" in self.titles[ax_id]:
            cmap = "afmhot"
        else:
            cmap = "gray"
        self.axs[ax_id].imshow(img,cmap=cmap,extent=self.ax_tick_labels)
        if marker_pos is not None:
            x,y = marker_pos
            self.axs[ax_id].scatter(x,y,**self.markerprops)
        
        if marker_pos_2 is not None:
            x,y = marker_pos_2
            self.axs[ax_id].scatter(x,y,**self.markerprops_2)
        if redraw:
            self.draw()

    
    def on_mouse_clicked(self, event):
        """Callback to allow setting postion of marker one by clicking the plot"""
        if event.xdata != None and event.ydata != None:
            print(event.xdata, event.ydata)
            self.marker_pos = event.xdata, event.ydata
            for id_ in range(len(self.axs)):
                self.update(self.get_array(id_), self.marker_pos, ax_id=id_)
            if self._on_marker_pos_changed is not None:
                self._on_marker_pos_changed()

    def draw(self):
        """Update whole figure."""
        try:
            self.fig.canvas.draw()
        except AttributeError:
            pass
    
    def get_array(self, ax_id:int=0) -> np.ndarray:
        """Return plotted data as numpy array"""
        return self.axs[ax_id].images[0].get_array()

class DriftCorrection:
    """
    Performs 2D AFM scans. A scan can be either a base_img or target_img.
    Every target_img is compared to the current base_img and the relative 
    shift of both images is calculated with cross correlation.

    Args:
        plot (Plot): plotted live data where data are retrieved from
        channel (str): channel name to observe
        res (int): number of pixels of test scans
        length (float): physical length of test scans
        ms_px (float): integration time per pixel
        line_level (bool): apply line levelling if Z channel is used for comparison
        plane_correction (bool): apply plane correction if Z channel is used for comparison
    """
    ms_px = 5.3
    res_x = 50
    res_y = 50
    len_x = 0.5
    len_y = 0.5
    angle = 0
    name = "Drift Scan"


    def __init__(self, plot, channel = "M1A", res = 50, length = 0.5, ms_px = 5.3, line_level = True, plane_correction = True) -> None:
        self.base_img = None
        self.target_img = None
        self.channel = channel
        self.all_spectra = []
        self.all_drifts = []
        self.fourier_parameters = None
        self.base_offset = None
        self.start_img = None
        self.line_level = line_level
        self.plane_correction = plane_correction
        self.res_x = self.res_y = int(res)
        self.len_x = self.len_y = float(length)
        self.ms_px = float(ms_px)
        self.plot = plot
    
    def do_drift_scan(self, offset_x, offset_y):
        """
        Perform a new AFM scan.
        If a base_img has already been set, the new scan is compared
        against it. If there is no base_img, the new scan is set to
        be the new base_img.
        Scan data are not saved in the database of the neaSCOPE controller.
        """
        with scan.Afm(self.name,
                      PhysicalOffsetX=offset_x,
                      PhysicalOffsetY=offset_y,
                      PhysicalSizeX=self.len_x,
                      PhysicalSizeY=self.len_y,
                      TargetResolutionWidth=self.res_x,
                      TargetResolutionHeight=self.res_y,
                      Angle=self.angle,TargetMillisecondsPerPixel=self.ms_px) as afm:
            # afm.scanparameters.ScanMode = nea.ScanMode.Serpent
            print("Starting Scan", end="\r")
            afm.scanparameters.ResolutionDepth = 1
            afm.scanparameters.DepthRuns = 1
            context.Microscope.Py.ScanAsync(afm.scanparameters)
            sleep(1)
            while context.Microscope.Py.RouteState != nea.RouteState.Idling:
                progress = context.Microscope.Py.RouteProgress
                t_remain = int(afm.scanparameters.Duration.TotalSeconds * (1-progress))
                min_ = t_remain//60
                sec = t_remain%60
                if sec < 10:
                    print(f"Scanning: {int(progress*100)} %, {min_}:0{sec}     ",end="\r")
                else:
                    print(f"Scanning: {int(progress*100)} %, {min_}:{sec}     ",end="\r")
                sleep(0.3)
            sleep(0.3)
            target_img = self.plot.get_array(1)
        if isinstance(target_img,np.ma.MaskedArray):
            target_img = np.nan_to_num(target_img.data,nan = np.mean(target_img))
        if self.plane_correction and self.channel == "Z":
            target_img = plane_subst(target_img)
        if self.line_level and self.channel == "Z":
            target_img = line_diff(target_img)
        if self.base_img is not None:
            shift = get_shift(self.base_img,target_img,(self.len_x/self.res_x,self.len_y/self.res_y))
            self.target_img = target_img
        else:
            self.base_img = self.start_img = target_img
            shift = (0,0)
        return shift

    def set_parameters(self, res=None, length = None, ms_px = None):
        """Change scanning parameters resolution, physical length or integration time."""
        if res is not None:
            self.res_x = self.res_y = int(res)
        if length is not None:
            self.len_x = self.len_y = float(length)
        if ms_px is not None:
            self.ms_px = float(ms_px)
            
        

class ParticleTracker:
    """
    Main class for particle tracking. Combines retrieval of scan data via `nea_tools Preview` class,
    plotting these data with `Plot`, and performing AFM scan with `DriftCorrection`

    Args:
        channel (str): channel to use for image comparison
        res (int): number of pixels in scan
        length (float): physical length of scan
        ms_px (float): integration time used in scan.
    """
    def __init__(self, channel = "M1A", res = 50, length = 0.5, ms_px = 5.3):
        self.plot = Plot(num_plots = 3)
        self.drift_correction = DriftCorrection(self.plot, channel = channel, res = res, length = length, ms_px = ms_px)
        self.do_plot_preview = True
        self.plot.titles = ["Z", channel, channel]
        self.preview = Preview(callback=self.plot_preview, channels = ["Z","M1A","M1P",channel,"O2A"])
        self.preview.run()
        self.tip_position = None
        self.plot.axs[2].set_visible(False)
    
    def plot_preview(self,preview_data):
        """Plot new scan data which are given in form of `nea_tools.datatypes.PreviewData`"""
        try:
            self.plot.ax_tick_labels = [
                self.preview.scanparameters.PhysicalOffsetX-self.preview.scanparameters.PhysicalSizeX/2,
                self.preview.scanparameters.PhysicalOffsetX+self.preview.scanparameters.PhysicalSizeX/2,
                self.preview.scanparameters.PhysicalOffsetY+self.preview.scanparameters.PhysicalSizeY/2,
                self.preview.scanparameters.PhysicalOffsetY-self.preview.scanparameters.PhysicalSizeY/2,
            ]
        except AttributeError: # initial call during init of self.preview
            pass
        if self.do_plot_preview and preview_data.channel_name == self.drift_correction.channel  and preview_data.ndim == 2:
            self.plot.update(preview_data.image.reshape(preview_data.shape), self.plot.marker_pos, self.plot.marker_pos_2, ax_id = 1, redraw=False, title = self.drift_correction.channel)
        if self.do_plot_preview and preview_data.channel_name == "Z"  and preview_data.ndim == 2:
            self.plot.update(preview_data.image.reshape(preview_data.shape), self.plot.marker_pos, self.plot.marker_pos_2, ax_id = 0, title = "Z")

    def measure_base_image(self):
        """Record a new base image with current scan parameters."""
        self.drift_correction.base_img = None
        self.plot.marker_pos_2 = None
        self.tip_position = current_tip_position()
        try:
            #self.plot.axs[2].set_visible(False)
            self.drift_correction.do_drift_scan(*self.tip_position)
            self.plot.update(self.drift_correction.start_img,self.plot.marker_pos, self.plot.marker_pos_2,ax_id=2, title = f"Reference {self.drift_correction.channel}")
            self.plot.axs[2].set_visible(True)
        except KeyboardInterrupt:
            print("\nManual interrupt")
    
    def run_tracking(self, num_scans, tsleep = 10) -> typing.List[typing.Tuple[float,float]]:
        """
        Runs multiple scans one after another. Each scan is compared to the current base_img.

        Args:
            num_scans (int): number of scans to record
            tsleep (float): waiting time between two consecutive scans, in sec
        
        Returns:
            shifts: list of x,y coordinates pairs of the relative shift of each scan 
        """
        if self.plot.marker_pos is None:
            raise AttributeError("Reference marker position is not defined")
        if self.drift_correction.base_img is None:
            raise AttributeError("Reference image is no defined") 
        shifts = []
        cum_shift = 0,0
        try:
            for i in range(num_scans):
                print(f"RUN {i+1}", end= "\r")
                shift = self.drift_correction.do_drift_scan(*self.tip_position)
                cum_shift = cum_shift[0]+shift[0],cum_shift[1]+shift[1]
                shift_nm = f"({int(shift[0] * 1000)} nm, {int(shift[1] * 1000)} nm)"
                print(f"RUN {i+1} - shift (x,y) = {shift_nm}")
                self.plot.marker_pos_2 = self.plot.marker_pos[0]+shift[0], self.plot.marker_pos[1]+shift[1] 
                self.plot.update(self.drift_correction.target_img,self.plot.marker_pos, self.plot.marker_pos_2,ax_id=1, title = f"Recent {self.drift_correction.channel}")
                # self.plot.update(self.drift_correction.start_img,self.plot.marker_pos, self.plot.marker_pos_2,ax_id=0, title = f"Start {self.drift_correction.channel}")
                shifts.append(shift)
                start = time()
                i = 0
                while time() < start + tsleep:
                    sleep(1)
                    print("Waiting "+"."*(i%3+1)+"      ", end = "\r")
                    i += 1
        except KeyboardInterrupt:
            print("\nManual interrupt")
        else:
            return shifts

    def close(self):
        """Dispose cleanly"""
        self.preview.stop()
        self.plot.fig.clear()

    def set_channel(self, channel):
        """Set new channel for image comparison."""
        self.plot.titles = ["Z", channel]
        self.preview.channels = ["Z","M1A","M1P",channel,"O2A"] 

    def set_resolution(self, res):
        """Set scan resolution"""
        self.drift_correction.set_parameters(res = res)
        
    def set_ms_px(self, ms_px):
        """Set integration time"""
        self.drift_correction.set_parameters(ms_px = ms_px)
        
    def set_length(self, length):
        """Set physical size of scan"""
        self.drift_correction.set_parameters(length = length)

class ParametersSliders:
    """
    Adds UI elements to change scan parameters of ParticleTracker
    
    Args:
        particle_tracker (ParticleTracker): instance to be affected by these UI elements
    """
    def __init__(self, particle_tracker):
        self.particle_tracker = particle_tracker
        self.slider_res = widgets.IntSlider(
            value = particle_tracker.drift_correction.res_x, 
            min = 10,
            max = 100,
            step = 1,
            description = "Pixels",
            orientation = "horizontal",
            readout_format = "d"
        )
        self.slider_length = widgets.IntSlider(
            value = int(particle_tracker.drift_correction.len_x*1000), 
            min = 50,
            max = 1000,
            step = 50,
            description = "Length [nm]",
            orientation = "horizontal",
            readout_format = "d"
        )
        self.slider_ms_px = widgets.FloatSlider(
            value = particle_tracker.drift_correction.ms_px, 
            min = 0.4,
            max = 10.2,
            step = 0.4,
            description = "ms/px",
            orientation = "horizontal",
            readout_format = ".1f"
        )
        self.slider_res.observe(lambda change: self.particle_tracker.set_resolution(change["new"]), names='value')
        self.slider_length.observe(lambda change: self.particle_tracker.set_length(change["new"]/1000), names='value')
        self.slider_ms_px.observe(lambda change: self.particle_tracker.set_ms_px(change["new"]), names='value')
        self.display()
        
    def display(self):
        display(self.slider_res, 
                self.slider_length, 
                self.slider_ms_px
               )

## Particle Tracker Main

**1. Create Interface**

In [ ]:
try:
    tracker.close()
except NameError:
    pass
tracker = ParticleTracker(channel = "M1P", res = 60, length = 0.3, ms_px = 0.8)
ui = ParametersSliders(tracker)

**2. Measure reference image**

In [ ]:
tracker.measure_base_image()

**3. Run tracker**

In [ ]:
drift = tracker.run_tracking(20,1)

## Disconnect and close

In [ ]:
try:
    tracker.close()
except NameError:
    pass
disconnect()